# Step 11: public-sample-guided budget allocation for local DP (novelty experiment)
Question: at a FIXED total epsilon, does it help to spend the budget unevenly across attributes, using attribute importance estimated from the *public* reference sample (no privacy cost)?
Schemes (all have the same total epsilon = sum of per-attribute epsilons + label epsilon):
- **uniform**: epsilon/(d+1) for every attribute and the label (the scheme used so far)
- **prop**: label gets epsilon/(d+1); the rest is split across attributes in proportion to their mutual information with the label (computed on the public sample)
- **topk**: only the k=5 most informative attributes (by public-sample MI) are released, each with epsilon/(k+1); the others are not released at all (zero privacy loss)
Models: local LR and local Naive Bayes, trained on randomized data and tested on clean records. Central DP-LR is the reference.

In [ ]:
!pip -q install diffprivlib
import numpy as np, pandas as pd, warnings, time, os, glob, shutil
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import fetch_openml
from diffprivlib.models import LogisticRegression as DPLR, GaussianNB as DPGNB
warnings.filterwarnings("ignore")

try:
    from google.colab import drive, files
    drive.mount('/content/drive'); OUT = "/content/drive/MyDrive/dp_paper/final/"; IN_COLAB = True
except Exception:
    OUT = "./final/"; IN_COLAB = False
os.makedirs(OUT + "figures", exist_ok=True)

QUICK = os.environ.get("DP_QUICK") == "1"          # tiny smoke-test mode (ignore in Colab)
SEED = 42; MAX_ROWS = 5000; PUBLIC_FRAC = 0.10; NB_BINS = 5
TOTALS = [10, 25, 50, 100]                          # matched total epsilon
N_DRAWS = 5                                         # noise draws / model seeds
MIA_REPS = 10; REID_DRAWS = 3; POOLS = [100, 500, 2000]; REID_TOTALS = [10, 25, 50, 100]
if QUICK: TOTALS = [25, 50]; N_DRAWS = 2; MIA_REPS = 3; REID_DRAWS = 1; POOLS = [100, 500]; REID_TOTALS = [25, 50]

# optional extra datasets from CSV files you upload to Colab:  name: (path, target_column, positive_value)
# Lending Club (small, 9,578 loans, 2007-2010): download loan_data.csv from Kaggle, upload to Colab, then use:
#   EXTRA_CSV = {"lending_club": ("/content/loan_data.csv", "not.fully.paid", 1)}
EXTRA_CSV = {}

In [ ]:
# ---------------- dataset loaders ----------------
def load_loan():
    d = pd.read_csv("https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv")
    return d.drop(columns=["Loan_ID", "Loan_Status"]), (d.Loan_Status == "Y").astype(int).values
def load_german():
    g = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv", header=None)
    return g.iloc[:, :-1], (g.iloc[:, -1] == 2).astype(int).values
def load_adult():
    a = fetch_openml("adult", version=2, as_frame=True, parser="auto").frame
    return a.drop(columns=["class"]), (a["class"] == ">50K").astype(int).values
def load_taiwan():
    t = fetch_openml(data_id=42477, as_frame=True, parser="auto").frame
    return t.iloc[:, :-1], pd.factorize(t.iloc[:, -1])[0].astype(int)
def load_openml(i):
    f = fetch_openml(data_id=i, as_frame=True, parser="auto").frame
    t = f.columns[-1]; lab = f[t].astype(str)
    return f.drop(columns=[t]), (lab != lab.mode()[0]).astype(int).values
def load_lending_club():
    d = pd.read_csv("https://raw.githubusercontent.com/Amitabh1989/random_forest/master/loan_data.csv")   # Lending Club 2007-2010, 9,578 loans
    return d.drop(columns=["not.fully.paid"]), d["not.fully.paid"].astype(int).values
def load_openml_xy(i):
    b = fetch_openml(data_id=i, as_frame=True, parser="auto")        # uses the dataset's own target column
    X = b.data; lab = b.target.astype(str)
    return X, (lab != lab.mode()[0]).astype(int).values              # positive class = minority (default)
def load_csv(path, target, positive=None):
    d = pd.read_csv(path); d = d.loc[:, ~d.columns.astype(str).str.contains("^Unnamed")]
    lab = d[target]
    y = (lab == positive).astype(int).values if positive is not None else (lab.astype(str) != lab.astype(str).mode()[0]).astype(int).values
    return d.drop(columns=[target]), y

ALL = {"loan": load_loan, "german": load_german, "adult": load_adult, "taiwan": load_taiwan,
       "credit_approval": lambda: load_openml(29), "bank_marketing": lambda: load_openml(1461), "diabetes": lambda: load_openml(37),
       "gmc": lambda: load_openml_xy(46468),
       "lending_club": load_lending_club}          # Give Me Some Credit (OpenML 46468, 45,000 rows, ~6.7% default)
MAX_ROWS_OVERRIDE = {"gmc": 15000}
for k, (p, t, pos) in EXTRA_CSV.items(): ALL[k] = (lambda p=p, t=t, pos=pos: load_csv(p, t, pos))
DATASETS = {k: ALL[k] for k in (["loan", "german", "lending_club"] if QUICK else list(ALL))}

def prepare(name):
    X, y = DATASETS[name](); y = np.asarray(y).astype(int)
    cap = MAX_ROWS_OVERRIDE.get(name, MAX_ROWS)
    if len(X) > cap:
        idx = np.random.default_rng(SEED).choice(len(X), cap, replace=False); X, y = X.iloc[idx].reset_index(drop=True), y[idx]
    X = X.copy(); num, cat = [], []
    for c in X.columns:
        if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 10: X[c] = X[c].astype(float); num.append(str(c))
        else: X[c] = pd.factorize(X[c].astype(str))[0]; cat.append(str(c))
    X.columns = X.columns.astype(str)
    perm = np.random.default_rng(SEED).permutation(len(X)); n_pub = max(int(PUBLIC_FRAC * len(X)), 50)
    Xpub = X.iloc[perm[:n_pub]].reset_index(drop=True); Xpriv = X.iloc[perm[n_pub:]].reset_index(drop=True); ypriv = y[perm[n_pub:]]
    for c in num:
        m = Xpub[c].median(); Xpub[c] = Xpub[c].fillna(m); Xpriv[c] = Xpriv[c].fillna(m)
    bounds = {}
    for c in num:
        lo, hi = float(Xpub[c].quantile(.01)), float(Xpub[c].quantile(.99))
        bounds[c] = (lo, hi if hi > lo else lo + 1.0)
    dom = {c: int(X[c].max()) + 1 for c in cat}
    edges = {c: np.quantile(Xpub[c], np.linspace(0, 1, NB_BINS + 1)[1:-1]) for c in num}
    ypub = y[perm[:n_pub]]
    return dict(name=name, X=Xpriv, y=ypriv, Xpub=Xpub, ypub=ypub, num=num, cat=cat, bounds=bounds, dom=dom, edges=edges, d=len(num) + len(cat))

In [ ]:
# ---------------- mechanisms and designs ----------------
def scaled(X, D, c):
    lo, hi = D["bounds"][c]; return (X[c].clip(lo, hi).values - lo) / (hi - lo)       # in [0,1]; sensitivity 1
def onehot(codes, K):
    M = np.zeros((len(codes), K)); M[np.arange(len(codes)), np.asarray(codes).astype(int)] = 1; return M
def rr_codes(col, k, eps, rng):
    col = np.asarray(col).astype(int)
    if k == 1: return col
    p = np.exp(eps) / (np.exp(eps) + k - 1); keep = rng.random(len(col)) < p
    r = rng.integers(0, k - 1, len(col)); return np.where(keep, col, r + (r >= col))
def chan(k, eps):
    p = np.exp(eps) / (np.exp(eps) + k - 1); q = (1 - p) / (k - 1); M = np.full((k, k), q); np.fill_diagonal(M, p); return M

def design(D, eps=None, rng=None):
    # eps=None -> clean design; otherwise Laplace(1/eps) on scaled numerics + randomized response on categoricals
    X = D["X"]; cols, names = [], []
    for c in D["num"]:
        v = scaled(X, D, c)
        if eps is not None: v = v + rng.laplace(0, 1 / eps, len(v))
        cols.append(v[:, None]); names.append(c)
    for c in D["cat"]:
        K = D["dom"][c]; codes = X[c].values.astype(int)
        if eps is not None: codes = rr_codes(codes, K, eps, rng)
        cols.append(onehot(codes, K)); names += [f"{c}_{k}" for k in range(K)]
    return pd.DataFrame(np.hstack(cols), columns=names)

def cv_auc(D, fit_score):
    y = D["y"]; a = []
    for tr, te in StratifiedKFold(10, shuffle=True, random_state=SEED).split(D["X"], y):
        a.append(roc_auc_score(y[te], fit_score(tr, te)))
    return float(np.mean(a))

def central(D, Z, make, reps):
    y = D["y"]; v = [cv_auc(D, lambda tr, te, s=s: make(s).fit(Z.iloc[tr], y[tr]).predict_proba(Z.iloc[te])[:, 1]) for s in range(reps)]
    return np.mean(v), np.std(v)

def local_lr(D, Zc, eps, rng):
    y = D["y"]; Zn = design(D, eps, rng); yn = rr_codes(y, 2, eps, rng)      # label randomized too
    def fs(tr, te):
        m = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500)).fit(Zn.iloc[tr], yn[tr])
        return m.predict_proba(Zc.iloc[te])[:, 1]                           # clean test records
    return cv_auc(D, fs)

def local_nb(D, eps, rng):
    X = D["X"]; y = D["y"]
    Cc = {c: X[c].values.astype(int) for c in D["cat"]}
    for c in D["num"]: Cc[c] = np.clip(np.searchsorted(D["edges"][c], X[c].clip(*D["bounds"][c]).values), 0, NB_BINS - 1)
    K = {**D["dom"], **{c: NB_BINS for c in D["num"]}}
    Cn = {c: rr_codes(Cc[c], K[c], eps, rng) for c in Cc}; yn = rr_codes(y, 2, eps, rng)
    My = np.linalg.inv(chan(2, eps)); Mx = {c: np.linalg.inv(chan(K[c], eps)) for c in Cc if K[c] > 1}
    def fs(tr, te):
        ny = np.clip(My @ np.bincount(yn[tr], minlength=2), 1e-3, None); score = np.tile(np.log(ny / ny.sum()), (len(te), 1))
        for c in Cc:
            if K[c] == 1: continue
            N = np.zeros((2, K[c])); np.add.at(N, (yn[tr], Cn[c][tr]), 1)
            T = np.clip(My @ N @ Mx[c], 0, None) + 1.0; P = T / T.sum(1, keepdims=True)    # debiased joint counts + smoothing
            score += np.log(P[:, Cc[c][te]]).T
        return score[:, 1] - score[:, 0]
    return cv_auc(D, fs)

In [ ]:
# ---------------- allocation experiment ----------------
from sklearn.metrics import mutual_info_score
def attr_codes(Xd, D, c):
    if c in D["num"]: return np.clip(np.searchsorted(D["edges"][c], Xd[c].clip(*D["bounds"][c]).values), 0, NB_BINS - 1)
    return Xd[c].values.astype(int)
def public_mi(D):
    return {c: mutual_info_score(D["ypub"], attr_codes(D["Xpub"], D, c)) for c in D["num"] + D["cat"]}
def allocate(D, tot, scheme, mi, k=5):
    attrs = D["num"] + D["cat"]; d = len(attrs)
    if scheme == "uniform": e = tot / (d + 1); return {c: e for c in attrs}, e
    if scheme == "prop":
        ey = tot / (d + 1); w = np.array([mi[c] for c in attrs]) + 0.05 * max(mi.values()) + 1e-9
        return {c: (tot - ey) * wi / w.sum() for c, wi in zip(attrs, w)}, ey
    if scheme == "topk":
        sel = sorted(attrs, key=lambda c: -mi[c])[:min(k, d)]; e = tot / (len(sel) + 1); return {c: e for c in sel}, e
def design_sub(D, epsd, rng=None):
    X = D["X"]; cols, names = [], []
    for c in D["num"]:
        if c not in epsd: continue
        v = scaled(X, D, c)
        if rng is not None: v = v + rng.laplace(0, 1 / epsd[c], len(v))
        cols.append(v[:, None]); names.append(c)
    for c in D["cat"]:
        if c not in epsd: continue
        K = D["dom"][c]; codes = X[c].values.astype(int)
        if rng is not None: codes = rr_codes(codes, K, epsd[c], rng)
        cols.append(onehot(codes, K)); names += [f"{c}_{k}" for k in range(K)]
    return pd.DataFrame(np.hstack(cols), columns=names)
def lr_alloc(D, epsd, ey, rng):
    y = D["y"]; Zc = design_sub(D, epsd); Zn = design_sub(D, epsd, rng); yn = rr_codes(y, 2, ey, rng)
    def fs(tr, te):
        m = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500)).fit(Zn.iloc[tr], yn[tr]); return m.predict_proba(Zc.iloc[te])[:, 1]
    return cv_auc(D, fs)
def nb_alloc(D, epsd, ey, rng):
    X = D["X"]; y = D["y"]; attrs = list(epsd)
    Cc = {c: attr_codes(X, D, c) for c in attrs}; K = {c: (D["dom"][c] if c in D["cat"] else NB_BINS) for c in attrs}
    Cn = {c: rr_codes(Cc[c], K[c], epsd[c], rng) for c in attrs}; yn = rr_codes(y, 2, ey, rng)
    My = np.linalg.inv(chan(2, ey)); Mx = {c: np.linalg.inv(chan(K[c], epsd[c])) for c in attrs if K[c] > 1}
    def fs(tr, te):
        ny = np.clip(My @ np.bincount(yn[tr], minlength=2), 1e-3, None); score = np.tile(np.log(ny / ny.sum()), (len(te), 1))
        for c in attrs:
            if K[c] == 1: continue
            N = np.zeros((2, K[c])); np.add.at(N, (yn[tr], Cn[c][tr]), 1)
            T = np.clip(My @ N @ Mx[c], 0, None) + 1.0; P = T / T.sum(1, keepdims=True); score += np.log(P[:, Cc[c][te]]).T
        return score[:, 1] - score[:, 0]
    return cv_auc(D, fs)

def run_alloc(D):
    mi = public_mi(D); y = D["y"]; Z = design(D); B = np.sqrt(D["d"]); rows = []
    for t in TOTALS:
        m, s = central(D, Z, lambda sd: DPLR(epsilon=t, data_norm=B, C=1.0, random_state=sd), N_DRAWS)
        rows.append(dict(dataset=D["name"], scheme="central", model="DPLR", total_eps=t, auc=m, sd=s))
        for scheme in ["uniform", "prop", "topk"]:
            epsd, ey = allocate(D, t, scheme, mi)
            for mdl, fn in [("LR", lr_alloc), ("NB", nb_alloc)]:
                v = [fn(D, epsd, ey, np.random.default_rng(13 * k + t + (7 if mdl == "NB" else 0))) for k in range(N_DRAWS)]
                rows.append(dict(dataset=D["name"], scheme=scheme, model=mdl, total_eps=t, auc=np.mean(v), sd=np.std(v)))
    return rows

for name in DATASETS:
    path = OUT + f"alloc_{name}.csv"
    if os.path.exists(path): print(name, "already done"); continue
    try:
        t0 = time.time(); D = prepare(name); pd.DataFrame(run_alloc(D)).to_csv(path, index=False); print(name, "done in", round(time.time() - t0), "s")
    except Exception as ex:
        import traceback; traceback.print_exc(); print(name, "FAILED:", repr(ex))

In [ ]:
# ---------------- summary, statistics, figure ----------------
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300})
A = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "alloc_*.csv"))], ignore_index=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
for t in sorted(A.total_eps.unique()):
    g = A[A.total_eps == t].copy(); g["key"] = g.scheme + "_" + g.model
    tb = g.pivot_table(index="dataset", columns="key", values="auc"); print(f"\n=== AUC at total epsilon = {t} ==="); print(tb.round(3).to_string())
    tb.round(3).to_csv(OUT + f"alloc_table_eps{t}.csv")
print("\n=== Gain over uniform (AUC difference) and Wilcoxon, across datasets ===")
for t in sorted(A.total_eps.unique()):
    g = A[A.total_eps == t]
    for mdl in ["LR", "NB"]:
        base = g[(g.scheme == "uniform") & (g.model == mdl)].set_index("dataset").auc
        for sch in ["prop", "topk"]:
            x = g[(g.scheme == sch) & (g.model == mdl)].set_index("dataset").auc; d_ = (x - base).dropna()
            p = stats.wilcoxon(d_).pvalue if len(d_) >= 5 and (d_ != 0).any() else np.nan
            cen = g[g.scheme == "central"].set_index("dataset").auc; gap = (cen - base).dropna()
            closed = (d_ / gap.reindex(d_.index)).replace([np.inf, -np.inf], np.nan).median()
            print(f"eps={t:>3} {mdl} {sch:5s}: mean gain {d_.mean():+.3f}, improved on {(d_ > 0).sum()}/{len(d_)}, Wilcoxon p={p:.3f}, median share of central gap closed {closed:+.2f}")
sel = [t for t in [25, 50] if t in A.total_eps.unique()]; fig, axs = plt.subplots(1, len(sel), figsize=(3.8 * len(sel), 3.1), sharey=True); axs = np.atleast_1d(axs)
dsl = list(A.dataset.unique()); x = np.arange(len(dsl)); w = .2
for ax, t in zip(axs, sel):
    g = A[A.total_eps == t]
    for k, (mdl, sch, col) in enumerate([("NB", "prop", "#D55E00"), ("NB", "topk", "#E69F00"), ("LR", "prop", "#009E73"), ("LR", "topk", "#56B4E9")]):
        base = g[(g.scheme == "uniform") & (g.model == mdl)].set_index("dataset").auc; v = g[(g.scheme == sch) & (g.model == mdl)].set_index("dataset").auc
        ax.bar(x + (k - 1.5) * w, [(v - base).get(d_, np.nan) for d_ in dsl], w, color=col, label=f"{mdl} {sch}")
    ax.axhline(0, c="k", lw=.8); ax.set_xticks(x); ax.set_xticklabels(dsl, rotation=40, ha="right"); ax.set_title(f"Total ε = {t}", pad=4)
axs[0].set_ylabel("AUC gain over uniform allocation"); axs[0].legend(frameon=False, fontsize=6.5)
fig.tight_layout(); os.makedirs(OUT + "figures", exist_ok=True); fig.savefig(OUT + "figures/fig_allocation.png", bbox_inches="tight"); fig.savefig(OUT + "figures/fig_allocation.pdf", bbox_inches="tight"); plt.show()
A.to_csv(OUT + "alloc_all.csv", index=False)